In [ ]:
##########################
####   Exe1 - Part1   ####
##########################  
from googletrans import Translator

# Create a translator object
translator = Translator()

# Example text
text = "Hello, how are you?"

# Translate from English (en) to French (fr)
result = translator.translate(text, src="en", dest="fr")

# Print the output
print(f"{text} -> {result.text}")


Hello, how are you? -> Bonjour comment allez-vous?


In [14]:
##########################
####   Exe1 - Part2   ####
##########################
from typing import List, Dict, Any
from googletrans import Translator

def batch_translate(
    texts: List[str],
    src: str = "en",
    dest: str = "fr"
) -> List[Dict[str, Any]]:
    """
    Translate multiple texts with per-item error handling.

    Returns a list of dicts with:
      - original: original text
      - translated: translated text (None if failed)
      - src: source language code
      - dest: destination language code
      - success: bool
      - error: error message if any
    """
    translator = Translator()
    results: List[Dict[str, Any]] = []

    for t in texts:
        try:
            res = translator.translate(t, src=src, dest=dest)
            results.append({
                "original": t,
                "translated": res.text,
                "src": src,
                "dest": dest,
                "success": True,
                "error": None,
            })
        except Exception as e:
            # Handle failure for this one item, keep going
            results.append({
                "original": t,
                "translated": None,
                "src": src,
                "dest": dest,
                "success": False,
                "error": str(e),
            })

    return results

# --- Example usage ---
if __name__ == "__main__":
    sample = ["Hello, how are you?", "This is great!", "See you tomorrow."]
    out = batch_translate(sample, src="en", dest="fr")
    for r in out:
        print(r)


{'original': 'Hello, how are you?', 'translated': 'Bonjour comment allez-vous?', 'src': 'en', 'dest': 'fr', 'success': True, 'error': None}
{'original': 'This is great!', 'translated': "C'est super!", 'src': 'en', 'dest': 'fr', 'success': True, 'error': None}
{'original': 'See you tomorrow.', 'translated': 'À demain.', 'src': 'en', 'dest': 'fr', 'success': True, 'error': None}


In [15]:
##########################
####   Exe2 - Part1   ####
##########################
# Required imports
from googletrans import LANGUAGES, Translator

# Create a translator object
translator = Translator()

# Example text (German)
text = "Guten Morgen, wie geht es dir?"

# Detect language
detection = translator.detect(text)

# Print result
print(f"Detected language: {detection.lang} ({LANGUAGES[detection.lang]})")


Detected language: de (german)


In [16]:
##########################
####   Exe2 - Part2   ####
##########################
# pip install googletrans==4.0.0-rc1
from typing import List, Dict, Any, Optional
from googletrans import LANGUAGES, Translator

class LanguageProcessor:
    """
    Integrated language processing:
      - Language detection
      - Single-text translation
      - Batch translation
      - Multi-target translation
    Includes:
      - Input validation
      - Per-item error handling
      - Language code <-> name mapping
    """
    # Soft guard: googletrans requests larger texts; chunking can help in some cases
    _MAX_CHARS_PER_CALL = 4500

    def __init__(self) -> None:
        self.translator = Translator()
        self.languages = LANGUAGES                       # code -> english name
        self._name_to_code = {v.lower(): k for k, v in self.languages.items()}  # name -> code

    # ---------- Utilities ----------
    def is_valid_code(self, code: str) -> bool:
        return isinstance(code, str) and code.lower() in self.languages

    def code_for_name(self, name: str) -> Optional[str]:
        """Map 'french' -> 'fr' (None if unknown)."""
        return self._name_to_code.get(str(name).strip().lower())

    def name_for_code(self, code: str) -> Optional[str]:
        """Map 'fr' -> 'french' (None if unknown)."""
        return self.languages.get(str(code).strip().lower())

    def _chunk(self, text: str) -> List[str]:
        """Split very long texts to reduce API failures."""
        if not isinstance(text, str) or len(text) <= self._MAX_CHARS_PER_CALL:
            return [text]
        chunks, step = [], self._MAX_CHARS_PER_CALL
        for i in range(0, len(text), step):
            chunks.append(text[i:i+step])
        return chunks

    # ---------- Core features ----------
    def detect(self, text: str) -> Dict[str, Any]:
        """Language detection with validation & clear return."""
        if not isinstance(text, str) or not text.strip():
            return {"success": False, "lang": None, "name": None, "confidence": None,
                    "error": "Empty or non-string text."}
        try:
            d = self.translator.detect(text)
            name = self.name_for_code(d.lang)
            return {"success": True, "lang": d.lang, "name": name, "confidence": getattr(d, "confidence", None)}
        except Exception as e:
            return {"success": False, "lang": None, "name": None, "confidence": None, "error": str(e)}

    def translate(
        self,
        text: str,
        dest: str,
        src: Optional[str] = None,   # None/"" => auto-detect
    ) -> Dict[str, Any]:
        """Single text translation with auto-detect and chunk-join for long texts."""
        if not isinstance(text, str) or not text.strip():
            return {"success": False, "original": text, "translated": None,
                    "src": src, "dest": dest, "error": "Empty or non-string text."}

        # Normalize language codes/names
        if dest and not self.is_valid_code(dest):
            # allow names like "french"
            name_code = self.code_for_name(dest)
            if name_code is None:
                return {"success": False, "original": text, "translated": None,
                        "src": src, "dest": dest, "error": f"Invalid destination language: {dest}"}
            dest = name_code

        if src:
            if not self.is_valid_code(src):
                name_code = self.code_for_name(src)
                if name_code is None:
                    return {"success": False, "original": text, "translated": None,
                            "src": src, "dest": dest, "error": f"Invalid source language: {src}"}
                src = name_code
        else:
            src = "auto"

        try:
            # Chunk very long texts; join results
            pieces = []
            for chunk in self._chunk(text):
                pieces.append(self.translator.translate(chunk, src=src, dest=dest).text)
            translated = "".join(pieces)

            return {
                "success": True,
                "original": text,
                "translated": translated,
                "src": src,
                "dest": dest,
                "src_name": self.name_for_code(src) if src != "auto" else "auto",
                "dest_name": self.name_for_code(dest),
                "error": None,
            }
        except Exception as e:
            return {"success": False, "original": text, "translated": None,
                    "src": src, "dest": dest, "error": str(e)}

    def translate_many(
        self,
        texts: List[str],
        dest: str,
        src: Optional[str] = None
    ) -> List[Dict[str, Any]]:
        """Batch translate with per-item isolation."""
        results: List[Dict[str, Any]] = []
        for t in texts:
            results.append(self.translate(t, dest=dest, src=src))
        return results

    def translate_to_many(
        self,
        text: str,
        targets: List[str],
        src: Optional[str] = None
    ) -> List[Dict[str, Any]]:
        """Translate one source text into multiple target languages."""
        out: List[Dict[str, Any]] = []
        for dest in targets:
            out.append(self.translate(text, dest=dest, src=src))
        return out


# ---------------- Example usage ----------------
if __name__ == "__main__":
    lp = LanguageProcessor()

    # 1) Detect
    print(lp.detect("Guten Morgen, wie geht's?"))

    # 2) Single translate (auto-detect src)
    print(lp.translate("Bonjour tout le monde", dest="en"))          # dest by code
    print(lp.translate("Hello, how are you?", dest="french"))        # dest by name

    # 3) Batch translate
    batch = ["This is great!", "See you tomorrow.", ""]
    print(lp.translate_many(batch, dest="fr", src="en"))

    # 4) One-to-many targets
    print(lp.translate_to_many("Good night", targets=["fr", "de", "es"]))


{'success': True, 'lang': 'de', 'name': 'german', 'confidence': None}
{'success': True, 'original': 'Bonjour tout le monde', 'translated': 'Hello everyone', 'src': 'auto', 'dest': 'en', 'src_name': 'auto', 'dest_name': 'english', 'error': None}
{'success': True, 'original': 'Hello, how are you?', 'translated': 'Bonjour comment allez-vous?', 'src': 'auto', 'dest': 'fr', 'src_name': 'auto', 'dest_name': 'french', 'error': None}
[{'success': True, 'original': 'This is great!', 'translated': "C'est super!", 'src': 'en', 'dest': 'fr', 'src_name': 'english', 'dest_name': 'french', 'error': None}, {'success': True, 'original': 'See you tomorrow.', 'translated': 'À demain.', 'src': 'en', 'dest': 'fr', 'src_name': 'english', 'dest_name': 'french', 'error': None}, {'success': False, 'original': '', 'translated': None, 'src': 'en', 'dest': 'fr', 'error': 'Empty or non-string text.'}]
[{'success': True, 'original': 'Good night', 'translated': 'Bonne nuit', 'src': 'auto', 'dest': 'fr', 'src_name': 